In [38]:
model_repo_path = r"HuggingFaceTB/SmolLM2-135M"
data_set_path = r"databricks/databricks-dolly-15k"

In [43]:
import torch

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
device

'cpu'

In [44]:
torch.__version__

'2.14.1+cpu'

In [3]:
from datasets import load_dataset

dataset = load_dataset(path=data_set_path, cache_dir="data/")
dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'context', 'response', 'category'],
        num_rows: 15011
    })
})

In [4]:
import random
samples = dataset['train'][random.sample(range(len(dataset['train'])), k=5)]
samples

{'instruction': ['Tell me about the new premier of New South Wales',
  'Summarize the Challenger Sales Methodology for me',
  'List down some facts regarding the Republican Party of Arkansas from given text',
  'What are common methods for processing coffee?',
  'I have a small ball, what games can I play?'],
 'context': ['Christopher John Minns (born 17 September 1979) is an Australian politician serving as the 47th and current premier of New South Wales since March 2023. He has been the leader of the New South Wales branch of the Australian Labor Party (ALP) since June 2021 and a member of the New South Wales Legislative Assembly (MLA) for the division of Kogarah since 2015. He was previously the deputy mayor of Hurstville from 2007 to 2008.\nMinns was first elected at the 2015 New South Wales state election, and was elected unopposed as leader of the Labor Party in leadership election in June 2021, following the resignation of Jodi McKay. He led the party to victory at the 2023 stat

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_repo_path, cache_dir='tokenizer/')

In [6]:
print(tokenizer.vocab_size)

49152


In [7]:
tokenizer.pad_token = tokenizer.eos_token # using eos token as pad token

In [8]:
tokenizer(["Hi there", "Hi there, how are u?"], return_tensors="pt", tokenize=False, padding=True, truncation=True)

{'input_ids': tensor([[26843,   665,     0,     0,     0,     0,     0],
        [26843,   665,    28,   638,   359,   326,    47]]), 'attention_mask': tensor([[1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1]])}

# Target data input-output formats

```pyhton
input = {messages:[{role:system , content: You are a helpful chat assistant},
{role:user , content: instruction+context },
{role:assistant , content: output },]}
```


In [9]:
import random

def combine_instruction_n_context(example):
    example['instruction'] = example['instruction']+"\n"*random.randint(0,3)+example['context']
    return example

In [10]:
combined_input_dataset = dataset.map(combine_instruction_n_context)
combined_input_dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'context', 'response', 'category'],
        num_rows: 15011
    })
})

In [11]:
def encode_example(example):
    example['encoded_instruction'] = tokenizer(example['instruction'], return_tensors="pt")
    example['encoded_respone'] = tokenizer(example['response'], return_tensors="pt")
    return example

In [12]:
dataset = combined_input_dataset.map(encode_example, remove_columns=['category', 'context'])
dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'response', 'encoded_instruction', 'encoded_respone'],
        num_rows: 15011
    })
})

In [13]:
dataset_splits = dataset['train'].train_test_split(test_size=0.2, seed=75)
dataset_splits

DatasetDict({
    train: Dataset({
        features: ['instruction', 'response', 'encoded_instruction', 'encoded_respone'],
        num_rows: 12008
    })
    test: Dataset({
        features: ['instruction', 'response', 'encoded_instruction', 'encoded_respone'],
        num_rows: 3003
    })
})

In [14]:
print(dataset_splits['train'][0]['instruction'])
print("==="*10)
print(dataset_splits['train'][0]['response'])

When was The White Mandigos band formed?The White Mandingos are a rock supergroup from Woodstock, New York consisting of rapper Murs, former Rolling Stone journalist and MTV / VH1 producer Sacha Jenkins and Bad Brains' bassist Darryl Jenifer.

Biography
The band was formed in late 2012 when Jenkins met up at Jenifer's house in Woodstock to discover if there was anything in common between their respective favourite music genres. They considered their initial collaborations unimpressive, so Jenkins suggested collaborating with Murs, who provided lyrics.

Their first album, The Ghetto Is Tryna Kill Me was released in June 2013, and followed with a short tour of the eastern United States, including gigs in New York's New Museum, Boston and Washington DC. The album is a concept album around Tyrone White, a young black man from a New York City housing project, who subsequently obtains a recording contract and gets a white girlfriend. Jenifer and Jenkins have described the album Tommy by The 

In [15]:
tokenizer.pad_token

'<|endoftext|>'

In [16]:
tokenizer('<|endoftext|>'), tokenizer('<|im_start|>'), tokenizer('<|im_end|>'), 

({'input_ids': [0], 'attention_mask': [1]},
 {'input_ids': [1], 'attention_mask': [1]},
 {'input_ids': [2], 'attention_mask': [1]})

In [17]:
tokenizer

GPT2Tokenizer(name_or_path='HuggingFaceTB/SmolLM2-135M', vocab_size=49152, model_max_length=8192, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	0: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<repo_name>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("<reponame>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	5: AddedToken("<file_sep>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	6: AddedToken("<filename>", rstrip

In [18]:
chat_template = """{%- for message in messages %}
    {%- if message['role']=='user'%}
        {{- '<|im_start|>'+'User:'+message['content']+'<|im_end|>'+'\n'}}
    {%- elif message['role']=='assistant'%}
        {{- '<|im_start|>'+'Assistant:'+message['content']+'<|im_end|><|endoftext|>'}}
    {%- elif message['role']=='system'%}
            {{- '<|im_start|>'+'System:'+message['content']+'<|im_end|>'+'\n'}}
    {%- endif %}
    {%- endfor %}"""

tokenizer.chat_template = chat_template

In [19]:
example = {"messages":[{"role":"system",'content':'hi'},{"role":"user",'content':'hello'},{"role":"assistant",'content':'bye'}]}
print(tokenizer.apply_chat_template(example['messages'], tokenize=False), tokenizer.apply_chat_template(example['messages']))

<|im_start|>System:hi<|im_end|>
<|im_start|>User:hello<|im_end|>
<|im_start|>Assistant:bye<|im_end|><|endoftext|> {'input_ids': [1, 18403, 42, 6004, 2, 198, 1, 11126, 42, 28120, 2, 198, 1, 9519, 9531, 42, 25257, 2, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [20]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(model_repo_path, cache_dir="model/")

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [21]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 576)
    (layers): ModuleList(
      (0-29): 30 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=576, out_features=576, bias=False)
          (k_proj): Linear(in_features=576, out_features=192, bias=False)
          (v_proj): Linear(in_features=576, out_features=192, bias=False)
          (o_proj): Linear(in_features=576, out_features=576, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=576, out_features=1536, bias=False)
          (up_proj): Linear(in_features=576, out_features=1536, bias=False)
          (down_proj): Linear(in_features=1536, out_features=576, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((576,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((576,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((576,), eps=1e-05)
    (rotary_emb): Lla

In [22]:
SYSTEM_PROMPT = """You are an helpful assistant. 
Your name is 'AARSON V1'. You follow user instructions clearly. 
You will use your knowledge and the provided context i.e., if there is any from the user, to help the user as much as you can."""

In [23]:
def pre_process_data(example):

    example['example'] = {'messages':[{'role':'system', 'content':SYSTEM_PROMPT},
                           {'role':'user', 'content':example['instruction']},
                           {'role':'assistant', 'content':example['response']}]}
    return example

In [24]:
dataset = dataset_splits.map(pre_process_data)
dataset

DatasetDict({
    train: Dataset({
        features: ['instruction', 'response', 'encoded_instruction', 'encoded_respone', 'example'],
        num_rows: 12008
    })
    test: Dataset({
        features: ['instruction', 'response', 'encoded_instruction', 'encoded_respone', 'example'],
        num_rows: 3003
    })
})

In [25]:
dataset['train'][0]['response'], dataset['train'][0]['example']

("The White Mandingos was formed in late 2012, and their first album 'The Guetto is Tryna Kill Me' was released in June 2013.",
 {'messages': [{'role': 'system',
    'content': "You are an helpful assistant. \nYour name is 'AARSON V1'. You follow user instructions clearly. \nYou will use your knowledge and the provided context i.e., if there is any from the user, to help the user as much as you can."},
   {'role': 'user',
    'content': 'When was The White Mandigos band formed?The White Mandingos are a rock supergroup from Woodstock, New York consisting of rapper Murs, former Rolling Stone journalist and MTV / VH1 producer Sacha Jenkins and Bad Brains\' bassist Darryl Jenifer.\n\nBiography\nThe band was formed in late 2012 when Jenkins met up at Jenifer\'s house in Woodstock to discover if there was anything in common between their respective favourite music genres. They considered their initial collaborations unimpressive, so Jenkins suggested collaborating with Murs, who provided lyr

In [26]:
sample = dataset['train'][0]['example']
inputs = tokenizer.apply_chat_template(sample['messages'], tokenize=True)['input_ids'][:-1]
labels = tokenizer.apply_chat_template(sample['messages'], tokenize=True)['input_ids'][1:]
print(tokenizer.decode(inputs))
print("-"*20)
print(tokenizer.decode(labels))

<|im_start|>System:You are an helpful assistant. 
Your name is 'AARSON V1'. You follow user instructions clearly. 
You will use your knowledge and the provided context i.e., if there is any from the user, to help the user as much as you can.<|im_end|>
<|im_start|>User:When was The White Mandigos band formed?The White Mandingos are a rock supergroup from Woodstock, New York consisting of rapper Murs, former Rolling Stone journalist and MTV / VH1 producer Sacha Jenkins and Bad Brains' bassist Darryl Jenifer.

Biography
The band was formed in late 2012 when Jenkins met up at Jenifer's house in Woodstock to discover if there was anything in common between their respective favourite music genres. They considered their initial collaborations unimpressive, so Jenkins suggested collaborating with Murs, who provided lyrics.

Their first album, The Ghetto Is Tryna Kill Me was released in June 2013, and followed with a short tour of the eastern United States, including gigs in New York's New Muse

In [27]:
from torch.utils.data import Dataset, DataLoader

class MyDataset(Dataset):
    def __init__(self, data, tokenizer):
        super().__init__()
        self.data = data
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        model_input = self.data[idx]['example']
        model_label = self.data[idx]['response']+'<|im_end|><|endoftext|>'

        tokenized_input = self.tokenizer.apply_chat_template(model_input['messages'], tokenize=True, return_tensors="pt")
        tokenized_output = self.tokenizer(model_label, tokenize=True, return_tensors='pt')
        return {'input_ids':tokenized_input['input_ids'].squeeze()[:-1],
                'attention_mask':tokenized_input['attention_mask'].squeeze()[:-1],
                'label':tokenized_output
                }

train_dataset = MyDataset(dataset['train'], tokenizer)
eval_dataset = MyDataset(dataset['test'], tokenizer)

In [28]:
def custm_collate_function(batch):
    input_ids = [item['input_ids'] for item in batch]
    attention_mask = [item['attention_mask'] for item in batch]
    label_input_ids = [item['label']['input_ids'].squeeze() for item in batch]
    label_attention_mask = [item['label']['attention_mask'].squeeze() for item in batch]

    padded_inputs = tokenizer.pad({'input_ids':input_ids, 'attention_mask':attention_mask}, return_tensors='pt')
    batch_sample = padded_inputs['input_ids'][0]
    print(batch_sample.shape)

    padded_labels = tokenizer.pad({'input_ids':label_input_ids, 'attention_mask':label_attention_mask}, return_tensors='pt', padding='max_length', padding_side='left', max_length=batch_sample.shape[0])
    print(padded_inputs['input_ids'].shape)
    print(padded_labels['input_ids'].shape)
    return padded_inputs.to(device), padded_labels.to(device)


In [29]:
ex = dataset['train'][0]
inp = tokenizer.apply_chat_template(ex['example']['messages'], tokenize=False, return_tensors="pt")
print(inp)
print(ex['response']+'<|im_end|><|endoftext|>')

<|im_start|>System:You are an helpful assistant. 
Your name is 'AARSON V1'. You follow user instructions clearly. 
You will use your knowledge and the provided context i.e., if there is any from the user, to help the user as much as you can.<|im_end|>
<|im_start|>User:When was The White Mandigos band formed?The White Mandingos are a rock supergroup from Woodstock, New York consisting of rapper Murs, former Rolling Stone journalist and MTV / VH1 producer Sacha Jenkins and Bad Brains' bassist Darryl Jenifer.

Biography
The band was formed in late 2012 when Jenkins met up at Jenifer's house in Woodstock to discover if there was anything in common between their respective favourite music genres. They considered their initial collaborations unimpressive, so Jenkins suggested collaborating with Murs, who provided lyrics.

Their first album, The Ghetto Is Tryna Kill Me was released in June 2013, and followed with a short tour of the eastern United States, including gigs in New York's New Muse

In [30]:
inp[-len(ex['response']+'<|im_end|><|endoftext|>'):] == ex['response']+'<|im_end|><|endoftext|>'

True

In [31]:
from transformers.data.data_collator import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding=True, return_tensors='pt')

BATCH_SIZE = 16

train_data_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=custm_collate_function)
eval_data_loader = DataLoader(eval_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=custm_collate_function)

In [32]:
next(iter(train_data_loader))

torch.Size([850])
torch.Size([16, 850])
torch.Size([16, 850])


({'input_ids': tensor([[    1, 18403,    42,  ...,     0,     0,     0],
         [    1, 18403,    42,  ...,     0,     0,     0],
         [    1, 18403,    42,  ...,     0,     0,     0],
         ...,
         [    1, 18403,    42,  ...,     0,     0,     0],
         [    1, 18403,    42,  ...,  2517,    30,     2],
         [    1, 18403,    42,  ...,     0,     0,     0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         ...,
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 1, 1,  ..., 0, 0, 0]])},
 {'input_ids': tensor([[    0,     0,     0,  ...,    30,     2,     0],
         [    0,     0,     0,  ...,    30,     2,     0],
         [    0,     0,     0,  ...,    30,     2,     0],
         ...,
         [    0,     0,     0,  ...,    89,     2,     0],
         [    0,     0,     0,  ...,    30,     2,     0],
         [    0,     0,     0,  ..., 18242,     2

In [33]:
for idx, d in enumerate(train_data_loader):
    print(d[0]['input_ids'][0])
    print()
    print(d[1]['input_ids'][0])
    print("="*30)
    if idx == 5:
        break

torch.Size([850])
torch.Size([16, 850])
torch.Size([16, 850])
tensor([    1, 18403,    42,  2683,   359,   354,  5356, 11173,    30,   216,
          198,  7334,  1462,   314,   637,    49,  2285, 13994,   717,    33,
         4289,  1206,  1066,  2914,  6388,  4763,    30,   216,   198,  2683,
          523,   722,   469,  1967,   284,   260,  2711,  2468,  2056,    30,
           85,  1143,   585,   665,   314,   750,   429,   260,  2914,    28,
          288,   724,   260,  2914,   347,  1083,   347,   346,   416,    30,
            2,   198,     1, 11126,    42,  2427,   436,   378,  5683, 12679,
          324,   395,  5975,  4331,    47,   504,  5683, 12679,   274,   395,
          359,   253,  4022,  2076,  5606,   429,  7817, 14621,    28,  1315,
         2863, 11057,   282,   689,  3115,   372,  2786,    28,  4302, 45600,
        12662, 18951,   284, 22446,    70,  2272,   717,    56,    33, 16741,
        39756,    81, 30726,   284, 13878,  4912,   927,    23, 17229,   363,
  

In [34]:
single_batch = next(iter(train_data_loader))
padded_inputs, padded_labels = single_batch

torch.Size([850])
torch.Size([16, 850])
torch.Size([16, 850])


In [35]:
model = model.to(device)
model.device

device(type='cpu')

In [36]:
padded_inputs['input_ids'].shape, padded_labels['input_ids'].shape

(torch.Size([16, 850]), torch.Size([16, 850]))

In [37]:
preds = model(padded_inputs['input_ids'])
preds

KeyboardInterrupt: 

In [ ]:
def calculate_loss(loss_fn, preds_logits, labels, labels_attention_mask):
    labels[labels_attention_mask==0] = -100
    loss = loss_fn(preds_logits.view(-1, preds_logits.shape[-1]), labels.view(-1))
    return loss

In [ ]:
import torch
loss_fn = torch.nn.CrossEntropyLoss(reduction='none', ignore_index=-100)
optim = torch.optim.AdamW(params=model.parameters(), lr=1e-3)

In [ ]:
def step(model, inputs, labels, loss_fn, optim):
    model.train()
    preds = model(inputs['input_ids'])
    loss = calculate_loss(loss_fn, preds.logits, labels['input_ids'], labels['attention_mask'])
    loss = torch.mean(loss[loss!=0])
    print(loss.item())
    loss.backward()
    optim.step()

In [ ]:
def train(epochs, model, data_loader, loss_fn, optim):
    for _ in range(epochs):
        for inputs, labels in data_loader:
            step(model, inputs, labels, loss_fn, optim)

In [ ]:
epochs=1
train(epochs, model, train_data_loader, loss_fn, optim)

torch.Size([424])
torch.Size([2, 424])
torch.Size([2, 424])
12.4375
